# じっせん君コメントシステム

Google Driveの実践事例PDFにClaude APIでコメントを生成し、Gmail下書きとして保存します。

## 使い方
1. 各セルを上から順に実行してください
2. 最初のセルで認証とセットアップを行います
3. テスト件数を指定して少量で動作確認できます

In [ ]:
# === Cell 1: セットアップ ===
!pip install -q anthropic pdfplumber reportlab pypdf

# Google認証
from google.colab import auth
auth.authenticate_user()

# リポジトリをクローン（初回のみ）
import os
if not os.path.exists('AIComment'):
    !git clone https://github.com/YOUR_REPO/AIComment.git
os.chdir('AIComment')

print('セットアップ完了')

In [ ]:
# === Cell 2: 設定 ===
import os
from google.colab import userdata

# Colab Secretsから設定値を取得
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['DRIVE_FOLDER_ID'] = userdata.get('DRIVE_FOLDER_ID')
os.environ['SPREADSHEET_ID'] = userdata.get('SPREADSHEET_ID')

# テスト件数（本番時は0に変更）
TEST_COUNT = 5

print(f'設定完了: テスト件数={TEST_COUNT}')

In [ ]:
# === Cell 3: フォント準備 ===
from src.utils import ensure_fonts, setup_logging

logger = setup_logging()
ensure_fonts()
print('フォント準備完了')

In [ ]:
# === Cell 4: データ取得 ===
from src import drive_client, sheets_client

# スプレッドシートから未処理レコード取得
records = sheets_client.get_unprocessed_records()
print(f'未処理レコード: {len(records)}件')
for r in records[:5]:
    print(f'  - {r.clinic_name} / {r.person_name}')

# DriveからPDF一覧取得
pdf_files = drive_client.list_pdfs()
print(f'\nPDFファイル: {len(pdf_files)}件')
for f in pdf_files[:5]:
    print(f'  - {f["name"]}')

In [ ]:
# === Cell 5: テスト実行（1件ずつ処理） ===
from src.main import run

# テスト件数だけ処理（通常モード）
run(test_count=TEST_COUNT)

In [ ]:
# === Cell 6: 結果確認 ===
# 処理後のスプレッドシートステータスを確認
all_records = sheets_client.read_records()
for r in all_records:
    status_mark = '✓' if r.status == '完了' else '✗' if 'エラー' in r.status else '-'
    print(f'  [{status_mark}] {r.clinic_name} / {r.person_name} → {r.status}')

## Batchモード（本番用・50%割引）

以下のセルは400件一括処理用です。テストで品質確認後に使用してください。

In [ ]:
# === Cell 7: Batchモード実行 ===
from src.batch_main import run as batch_run

# 全件をBatch APIで処理
# batch_run(batch_mode=True, test_count=0)  # コメント解除で実行